# Save Loader

Load an EU5 save file via rakaly CLI, build resolved lookup tables, and explore country/culture/religion data.

**Prerequisites:** rakaly binary at `bin/rakaly/`, a `.eu5` save file, and optionally localisation files at `game-data/eu5/localization/english/`.

In [1]:
# Setup: add project root to path
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\PH\Documents\Claude\Projects\PDX Save Analyzer


In [2]:
# === CONFIGURATION — edit these paths ===
SAVE_FILE   = "saves/autosave.eu5"                      # Path to your .eu5 save
RAKALY_BIN  = "bin/rakaly/rakaly"                        # rakaly CLI binary
LOC_DIR     = "game-data/eu5/localization/english"       # Localisation dir (set to None to skip)

print(f"Save file : {SAVE_FILE} — exists: {os.path.exists(SAVE_FILE)}")
print(f"Rakaly    : {RAKALY_BIN} — exists: {os.path.exists(RAKALY_BIN)}")
print(f"Loc dir   : {LOC_DIR} — exists: {os.path.exists(LOC_DIR) if LOC_DIR else 'skipped'}")

Save file : saves/autosave.eu5 — exists: True
Rakaly    : bin/rakaly/rakaly — exists: True
Loc dir   : game-data/eu5/localization/english — exists: True


In [3]:
# Load the save
from toolbox.save_loader import load_save

save = load_save(SAVE_FILE, rakaly_bin=RAKALY_BIN, loc_dir=LOC_DIR, verbose=True)
print("\nSave loaded successfully.")

[save_loader] Parsing autosave.eu5 via rakaly...



Save loaded successfully.


[save_loader] Parsed OK.
[save_loader] Loading localisation from game-data\eu5\localization\english...
[save_loader] Loaded 86,507 localisation entries.


## Save Overview

In [4]:
# Basic metadata
print(f"Date           : {save.game_date}")
print(f"Game version   : {save.game_version}")
print(f"Multiplayer    : {save.is_multiplayer}")
print(f"Current age    : {save.current_age_name} ({save.current_age_key})")
print(f"Player name    : {save.player_name}")
print(f"Player country : {save.player_country_name} ({save.player_country_tag})")
print(f"Playthrough ID : {save.raw['metadata'].get('playthrough_id', '?')}")
print(f"\nLookup sizes:")
print(f"  Cultures     : {len(save.culture_index):,}")
print(f"  Religions    : {len(save.religion_index):,}")
print(f"  Country tags : {len(save.tag_index):,}")
print(f"  Localisation : {len(save.loc):,} entries")

Date           : 1481.7.1
Game version   : 1.1.10
Multiplayer    : True
Current age    : Discovery (age_3_discovery)
Player name    : Capitaine Erin
Player country : Upper Bavaria (WUR)
Playthrough ID : c832299a-d810-47d8-a21a-3e5e710c98d9

Lookup sizes:
  Cultures     : 2,083
  Religions    : 293
  Country tags : 2,432
  Localisation : 86,507 entries


## Player Country Data

In [5]:
# Player country resources
pc = save.player_country_data()
currency = pc.get("currency_data", {})
balance = pc.get("balance_history_2", {})

print(f"=== {save.player_country_name} ({save.player_country_tag}) ===")
print(f"\nCurrency data:")
for k, v in currency.items():
    delta = balance.get(k.title().replace('_', ''), balance.get(k, '?'))
    print(f"  {k:<25s} {v:>12}   (monthly: {delta})")

print(f"\nKey stats:")
for key in ["estimated_monthly_income", "last_month_gold_income", "max_manpower",
            "last_months_population", "great_power_rank", "primary_culture", "primary_religion"]:
    val = pc.get(key, "?")
    extra = ""
    if key == "primary_culture" and isinstance(val, int):
        extra = f"  → {save.resolve_culture_name(val)}"
    elif key == "primary_religion" and isinstance(val, int):
        extra = f"  → {save.resolve_religion_name(val)}"
    print(f"  {key:<40s} {val}{extra}")

=== Upper Bavaria (WUR) ===

Currency data:
  manpower                       3.37026   (monthly: 0.0719)
  gold                         297.82273   (monthly: 20.41467)
  stability                     22.30018   (monthly: 0.11829)
  war_exhaustion                 0.31592   (monthly: -0.07165)
  inflation                      0.00053   (monthly: 1e-05)
  prestige                      79.25418   (monthly: -0.6618)
  army_tradition                41.27643   (monthly: -0.00599)
  government_power               83.6735   (monthly: -0.41325)
  karma                        -62.31306   (monthly: 0.19526)
  religious_influence           45.87868   (monthly: 0.54981)
  purity                              60   (monthly: ?)
  righteousness                       90   (monthly: ?)

Key stats:
  estimated_monthly_income                 41.44218
  last_month_gold_income                   37.39496
  max_manpower                             4.3944
  last_months_population                   429.97208
  gr

## All Real Countries

In [6]:
# List all real countries sorted by great power rank
countries = save.all_real_countries()
countries.sort(key=lambda c: c[2].get("great_power_rank", 9999))

print(f"Total real countries: {len(countries)}\n")
print(f"{'Rank':<6} {'TAG':<6} {'Name':<30} {'Gold':>10} {'Manpower':>10} {'Pop':>10}")
print("-" * 78)
for cid, tag, data in countries[:30]:
    name = save.country_display_name(tag)
    curr = data.get("currency_data", {})
    rank = data.get("great_power_rank", "?")
    gold = curr.get("gold", 0)
    mp = curr.get("manpower", 0)
    pop = data.get("last_months_population", 0)
    print(f"{rank:<6} {tag:<6} {name:<30} {gold:>10.1f} {mp:>10.2f} {pop:>10.1f}")

Total real countries: 2429

Rank   TAG    Name                                 Gold   Manpower        Pop
------------------------------------------------------------------------------
1      CHI    China                             41098.7     277.64    76855.4
2      MAJ    Majapahit                         22120.6       6.66     5968.3
3      KOR    Goryeo                             2264.8      20.16     3858.0
4      ORI    Orissa                           119321.9       0.95     5433.1
5      KHM    Khmer                              3445.0       2.43     5052.7
6      FRA    France                             2006.2      22.19     8850.2
7      CAS    Castile                             639.1       0.01     6488.2
8      PAP    Rome                               5027.5       0.00     2350.1
9      VIJ    Vijayanagar                        6057.9      23.72     5202.0
10     HSL    Hoysala                            1399.4       0.04     8253.5
11     MAM    Egypt                

## Resolve IDs

Quick lookups for culture, religion, and country IDs.

In [7]:
# Resolve any culture/religion/country ID
# Change these values to look up different IDs

CULTURE_ID  = 1066
RELIGION_ID = 12
COUNTRY_TAG = "WUR"

print(f"Culture {CULTURE_ID}  → key: {save.resolve_culture(CULTURE_ID)!r}")
print(f"           → name: {save.resolve_culture_name(CULTURE_ID)!r}")
print()
print(f"Religion {RELIGION_ID} → key: {save.resolve_religion(RELIGION_ID)!r}")
print(f"           → name: {save.resolve_religion_name(RELIGION_ID)!r}")
print()
print(f"Country {COUNTRY_TAG!r} → display: {save.country_display_name(COUNTRY_TAG)!r}")

Culture 1066  → key: 'swabian'
           → name: 'Swabian'

Religion 12 → key: 'catholic'
           → name: 'Catholicism'

Country 'WUR' → display: 'Württemberg'


## Raw JSON Access

The full parsed JSON is available at `save.raw`. Navigate freely.

In [8]:
# Top-level keys and their types
for k, v in save.raw.items():
    if isinstance(v, dict):
        print(f"  {k:<35s} dict ({len(v)} keys)")
    elif isinstance(v, list):
        print(f"  {k:<35s} list ({len(v)} items)")
    else:
        print(f"  {k:<35s} {type(v).__name__}: {str(v)[:60]}")

  metadata                            dict (13 keys)
  start_of_day                        str: 1481.7.1
  current_age                         str: age_3_discovery
  speed                               int: 4
  random_seed                         int: 3432947563
  random_count                        int: 112850247
  variables                           dict (2 keys)
  language_manager                    dict (2 keys)
  ironman_manager                     dict (2 keys)
  great_power_manager                 dict (2 keys)
  road_network                        dict (1 keys)
  resolution_manager                  dict (1 keys)
  situation_manager                   dict (22 keys)
  dynamic_game_object_manager         list (0 items)
  institution_manager                 dict (1 keys)
  loan_manager                        dict (1 keys)
  construction_manager                dict (1 keys)
  tutorial_manager                    dict (1 keys)
  culture_manager                     dict (1 keys)
  holy